<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_29_catalog_detection_product_validation.ipynb">9.29 源表与检测产品验证</a></li>
        <li>下一节： <a href="9_31_archive_replay_case_study.ipynb">9.31 公开归档数据复盘</a></li>
    </ul>
</div>


## 9.30 最小可运行处理工作流：环境、数据清单与运行报告

第 9.28 节给出了最小可复现项目的目录结构，第 9.29 节讨论了检测产品如何进入可发布样本。还需要补上一个更接近真实工作的环节：这些对象怎样被一次实际运行连接起来。一个目录模板本身不会复现结果；一组脚本本身也不会说明结果可信。真正可审查的是一次运行事件：某一份输入数据，在某一套环境和参数下，经过一组命令，生成一批产品和验证报告，并留下明确结论。

本节把“最小可运行工作流”定义为一个轻量但完整的处理事件。它不要求引入复杂工作流系统，也不要求把大型数据塞进版本库。它要求六类对象被显式连接：数据清单、环境记录、配置文件、执行入口、产品清单和运行报告。对于小型教学项目，这套结构可以由几个脚本和文本文件完成；对于团队或批量项目，它可以平滑迁移到更完整的工作流系统。

这样安排的原因很实际。射电干涉项目经常需要多次重跑：改 flag、换权重、调整 mask、更新源搜索阈值、排除坏频段、换软件版本、重新下载归档产品。若每次运行没有身份记录，多个版本的图像、源表和报告会很快混在一起。最小可运行工作流的目标，就是让每一次运行都能回答：输入是否相同，环境是否可识别，参数是否改变，哪些产品重新生成，验证是否通过，哪些人工判断影响了结论。


### 9.30.1 从项目模板到运行事件

可复现项目描述的是结构，运行事件描述的是一次处理。可以把一次运行写成

$$
R_k = \left(D_k, E_k, \Theta_k, S_k, P_k, Q_k, H_k\right),
$$

其中 $D_k$ 是第 $k$ 次运行使用的数据清单，$E_k$ 是环境记录，$\Theta_k$ 是配置参数，$S_k$ 是执行脚本或依赖规则，$P_k$ 是输出产品，$Q_k$ 是验证结果，$H_k$ 是人工判断和例外说明。可复现性的核心，是未来能够重新识别这些对象，并解释它们如何共同产生科学结论。

这个表示式强调两件事。第一，产品不是孤立文件。一个图像文件必须绑定输入数据、成像参数、软件环境、命令日志和验证指标；一个源表必须绑定输入图像、检测阈值、模型/残差图、边缘规则和可靠性估计。第二，重跑不是简单覆盖旧产品。若 $\Theta_k$ 改变，新的运行应得到新的运行编号；若 $D_k$ 或 $E_k$ 改变，即使参数相同，也应在报告中清楚说明差异。

最小工作流的“最小”不在于步骤少，而在于没有不必要的隐式状态。输入来自清单，环境来自记录，参数来自配置，执行入口固定，产品有清单，验证有报告。这样处理一次数据时，工作流不会因为个人机器路径、交互变量或手动覆盖文件而失去身份。


![最小可运行处理栈](figures/practical_runnable_workflow_run_stack.png)

图中把一次最小可重跑处理拆成科学目标、数据清单、环境记录、配置文件、执行脚本、产品目录和验证报告。下方每一类“身份”都被绑定到同一个运行编号。这里的重点不是自动化程度，而是让一次运行成为可审查的处理事件。


### 9.30.2 数据清单与校验：大数据不进版本库，身份必须进版本库

大型 Measurement Set、ASDM、FITS cube 和中间图像通常不适合直接进入 Git。它们体积大、变化频繁，也可能受到访问权限限制。但数据身份必须进入项目文本。最小数据清单应记录项目编号、观测日期、execution block、field、spectral window、scan、数据来源、下载日期、pipeline 产品版本、文件大小和校验和。归档数据若有多个版本，还应记录选择了哪一版以及为什么。

一个数据清单可以采用如下形式：

```yaml
data_set:
  project_code: EXAMPLE-2026.1.00001.S
  target_field: target_a
  execution_blocks:
    - uid: uid___A002_Xexample_X001
      date: 2026-03-14
      array_config: compact
      intent: science_target
  spectral_windows:
    - id: 17
      frequency_range: 1.30-1.55 GHz
      channel_width: 31.25 kHz
  download:
    source: archive_pipeline_product
    date: 2026-05-04
    pipeline_version: pipeline_2026.1
  files:
    - name: target_a_calibrated.ms.tar
      bytes: 18432000000
      checksum: sha256:examplehash
```

校验和的作用不是装饰。若输入文件缺失，处理应停止；若文件存在但校验不符，应重新下载或重新生成清单；若清单被更新，应记录版本和原因；只有校验通过的数据才进入处理。这个原则能避免非常隐蔽的错误：同名文件来自不同下载批次、归档产品被重新发布、局部拷贝损坏、旧 flag 版本被误用。

在教学案例中，数据可以是轻量仿真或小型公开 cutout；在研究项目中，数据可以位于共享存储。无论数据规模如何，清单、校验和和下载记录都应进入版本控制，因为它们定义了后续处理的输入身份。


![数据清单与校验](figures/practical_runnable_workflow_data_manifest_checksum.png)

图中展示数据清单、校验文件、数据存储和运行入口之间的关系。大型数据可以留在存储系统，项目文本保存身份。若文件缺失或校验不符，处理应停止；若清单更新，必须记录版本。


### 9.30.3 环境记录：分层保存，而不是只写版本号

软件环境很容易被低估。射电干涉处理的结果可能受到多个层级影响：操作系统、容器或模块系统、CASA/WSClean/DP3/DDFacet/PyBDSF/SoFiA 等射电软件、Python 程序包、项目脚本、配置文件。只记录“使用某版本 CASA”不能完整解释结果差异，因为底层库、默认参数、Python 依赖和项目脚本同样可能改变。

环境记录不必从一开始就极端复杂，但应区分三类层级。第一类是必须入库的文本层，包括项目脚本、配置文件、环境摘要和关键命令输出。第二类是建议锁定的运行层，包括 Python 环境文件、容器镜像名、模块集合和关键软件版本。第三类是可替换但需记录的系统层，包括操作系统、硬件架构、存储路径和编译环境。

一个轻量环境摘要可以包含：

```yaml
environment:
  run_platform:
    os: linux
    architecture: x86_64
  radio_tools:
    casa: 6.x
    wsclean: 3.x
    pybdsf: 1.x
    sofia: 2.x
  python_packages:
    astropy: recorded
    numpy: recorded
    scipy: recorded
    radio_beam: recorded
  project_code:
    git_commit: recorded
    config_hash: recorded
```

若项目进入长期发布阶段，容器或锁定依赖文件会更稳健；若只是课堂小项目，环境摘要加上可安装依赖文件已经能显著提高可复现性。关键不是所有层都固定不变，而是每一层的身份、可替换性和风险都被记录。


![环境记录分层](figures/practical_runnable_workflow_environment_layers.png)

图中把环境记录分为操作系统与硬件、容器或模块、射电软件、程序包环境、项目脚本和配置文件。越靠近项目科学判断的层级越应进入版本控制；越靠近系统基础设施的层级越需要记录可替换性和风险。


### 9.30.4 执行入口：从手动命令到薄脚本，再到依赖规则

执行入口应尽量薄。它不应把科学参数硬编码在脚本深处，而应读取配置文件，生成或调用相应软件命令，并把命令、退出状态、运行时间和日志保存下来。这样脚本的职责是“让步骤可重复”，配置文件的职责是“说明为什么这样处理”。

一个小项目可以从固定命名的脚本开始：

```text
scripts/
  check_inputs.sh
  make_environment_record.sh
  run_imaging.sh
  run_catalog.sh
  run_validation.sh
  write_run_report.sh
```

当产品数量增加后，可以把依赖写成轻量构建规则。规则的思想比具体工具更重要：产品由哪些输入、配置和脚本决定，哪些文件改变后必须重跑。一个简化规则可以表达为：

```text
products/images/target_image.fits:
    data/manifest.yml
    configs/imaging.yml
    scripts/run_imaging.sh

validation/noise_report.yml:
    products/images/target_image.fits
    configs/validation.yml
    scripts/run_validation.sh
```

这类规则可以由 Makefile、Snakemake、Nextflow、CWL 或自定义小脚本承载。对小型教学项目，薄脚本通常已经足够；当依赖关系、多目标、多频段和批量运行变复杂时，更完整的工作流系统才有必要。无论采用哪种执行方式，都应保留输入清单、环境记录、配置文件、命令日志、产品清单和验证报告。


![执行方式层级](figures/practical_runnable_workflow_execution_modes.png)

图中把执行方式分成手动命令、薄脚本、轻量构建规则和工作流系统。执行方式可以逐级增长，但核心依赖关系应保持不变。小项目从薄脚本开始通常更稳健，依赖和批量运行复杂后再引入更重的系统。


### 9.30.5 运行报告：把一次处理变成证据包

运行报告是最小工作流的出口。它不需要很长，却必须能回答四个问题：输入是否相同，环境是否可识别，参数是否改变，产品是否通过验证。报告应由脚本自动生成主要字段，并允许保留人工判断和例外说明。

一个运行报告可以包含如下结构：

```yaml
run_report:
  run_id: run_2026_05_04_001
  input:
    manifest_version: recorded
    checksum_status: passed
  environment:
    environment_record: records/environment_run_001.yml
    git_commit: recorded
  parameters:
    config_hash: recorded
    changed_since_previous_run: imaging_weighting
  execution:
    steps:
      - check_inputs: passed
      - imaging: passed
      - catalog: passed
      - validation: review_required
  products:
    images: products/images/target_image.fits
    catalogs: products/catalogs/target_catalog.fits
    residuals: products/images/target_residual.fits
  validation:
    image_rms: within_tolerance
    residual_structure: review_required
    flux_scale: passed
  decision:
    status: candidate
    next_action: inspect_residual_near_bright_source
```

报告中的 `decision` 字段尤其重要。真实项目并不总是得到简单通过或失败。某次运行可能适合生成候选图，尚不适合发布源表；某个源表可能适合内部交叉匹配，但不适合最终统计；某个 cube 可能可以给出上限，但不能声称检测。运行报告让这些中间状态有明确身份，避免后续误把候选产品当作最终产品。


![运行报告证据包](figures/practical_runnable_workflow_run_report_bundle.png)

图中把运行报告看成证据包中心。输入摘要、环境摘要、参数摘要、执行摘要、产品摘要、验证摘要、人工判断和发布状态共同决定一次处理的结论。报告不必冗长，但必须能说明结果是否可审查。


### 9.30.6 失败、部分重跑与陈旧产品

最小工作流应正视失败，而不是只记录成功路径。失败大致分为四类。第一类是输入失败，例如文件缺失、校验不符、清单与实际数据不一致。第二类是环境失败，例如软件不可用、版本不匹配、依赖包缺失。第三类是执行失败，例如命令退出、磁盘空间不足、中间产品损坏。第四类是验证失败，例如噪声过高、残差结构明显、通量恢复不合格、源表伪检过多。

这些失败的处理方式不同。输入失败和环境失败通常应在真正处理前停止；执行失败应保留日志和部分输出，但不能让下游步骤误用不完整产品；验证失败不一定说明运行无效，它可能说明需要调整 flag、权重、mask、源搜索阈值或科学结论。最危险的是陈旧产品：上游配置已经改变，下游文件却仍来自旧运行。依赖规则、产品清单和运行编号正是为了解决这个问题。

一个稳健实践是让每个运行写入独立目录，或至少让产品文件携带运行编号。若项目选择覆盖固定产品名，也应保留运行报告、配置哈希和产品清单，使每个输出都能追溯到具体运行。这样可以降低“看起来是最新结果，实际来自旧参数”的风险。


### 9.30.7 教学案例：一个归档连续谱数据的最小重跑包

设想一个归档 VLA 或 MeerKAT 连续谱数据集，pipeline weblog 显示基础校准通过，但新的科学目标需要更保守的 taper 和更严格的源表验证。一个最小重跑包可以这样组织：

```text
configs/
  imaging.yml
  catalog.yml
  validation.yml
records/
  data_manifest.yml
  checksums.txt
  environment_run_001.yml
  run_report_001.yml
scripts/
  check_inputs.sh
  run_imaging.sh
  run_catalog.sh
  run_validation.sh
  write_run_report.sh
products/
  images/
  catalogs/
  validation/
```

运行从 `check_inputs.sh` 开始。若清单和校验通过，`run_imaging.sh` 读取 `configs/imaging.yml`，生成成像命令并保存日志；`run_catalog.sh` 读取 `configs/catalog.yml`，生成源表、模型图和残差图；`run_validation.sh` 读取 `configs/validation.yml`，计算图像 RMS、残差峰值、负源统计和简单通量检查；最后 `write_run_report.sh` 把输入、环境、参数、产品和验证结果写入运行报告。

这个案例的教学重点，是把“能跑”与“可信”区分开。成像命令正常退出，只说明执行成功；图像噪声、残差和通量恢复通过验证，才说明产品可能适合当前科学目标；源表生成成功，只说明软件输出了表格；负源统计、边缘排查和残差检查通过后，源表才有发布意义。若验证报告给出候选状态，后续运行应从明确的配置改动开始，而不是凭记忆修改命令。


### 9.30.8 本节结论

最小可运行处理工作流把项目模板推进到运行层面。它要求数据清单定义输入身份，环境记录定义软件身份，配置文件定义参数身份，执行入口定义命令身份，产品清单定义输出身份，验证报告定义科学判断。只有这些对象被同一个运行编号连接起来，一次处理才真正可审查。

这套做法的价值不在于形式复杂，而在于降低真实研究中的混乱：同名数据来自不同下载，旧参数生成的新图像，交互命令缺少日志，验证失败被最终产品掩盖，候选产品被误当成发布样本。第 9.30 节与 9.28、9.29 一起，把软件生态、项目结构、检测产品和运行报告串成一条更完整的实践训练链。
